In [11]:
import requests
import json
import time
from pathlib import Path
import re


In [28]:
def load_json_file(filepath):
    """Load JSON data from a file."""
    try:
        with open(filepath, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: Could not find file {filepath}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON in file {filepath}")
        return None

def GetGazePathsFromFile(file_path): # Gives an array containing arrays of gaze points tupled with an array of lists of the most likely words for each
    #Returns: ([lists of points], [lists of words])
    try:
        # Open and read the file
        gazePaths = []
        topWords = []
        with open(file_path, 'r') as file:
            content = file.readlines()
        for line in content:
            # print(line)
            if (line.strip().startswith('{')):
                corrected_content = correct_json(line)
                # print(corrected_content)
                data = json.loads(corrected_content)
                if "top_words" in data:
                    topWords.append(data["top_words"])
                if "gaze_points" in data:
                    points = data["gaze_points"]
                    points = [(point['x'], point['y'], point['z']) for point in points]
                    gazePaths.append(points)
        return gazePaths, topWords
        # for i, path in enumerate(gazePaths):
        #     print(i)
    except FileNotFoundError:
        print(f"Error: File not found at path: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in the file: {str(e)}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {str(e)}")
        return None

def correct_json(content):
    # Replace single quotes with double quotes
    content = content.replace("'", '"')
    
    # Correct boolean values
    content = content.replace('True', 'true').replace('False', 'false')
    
    # Remove trailing commas in lists and objects
    content = re.sub(r',\s*}', '}', content)
    content = re.sub(r',\s*]', ']', content)
    
    # Wrap the entire content in curly braces if it's not already
    if not content.strip().startswith('{'):
        content = '{' + content + '}'
    
    return content

def parse_gaze_and_predictions(file_path):
    """
    Parse a file containing alternating JSON objects of gaze points and word predictions.
    Handles potential formatting issues in the JSON data.
    
    Args:
        file_path (str): Path to the JSON file
        
    Returns:
        tuple: (gaze_points_list, predictions_list) where:
            - gaze_points_list is a list of lists containing gaze point data
            - predictions_list is a list of lists containing word predictions
    """
    import json
    
    gaze_points_list = []
    predictions_list = []
    
    with open(file_path, 'r') as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue
                
            try:
                # Replace single quotes with double quotes to ensure valid JSON
                line = line.replace("'", '"')
                
                # Try to parse the JSON
                data = json.loads(line)
                
                # If the JSON object contains gaze_points, add to gaze_points_list
                if 'gaze_points' in data:
                    gaze_points_list.append(data['gaze_points'])
                
                # If the JSON object contains top_words, add to predictions_list
                elif 'top_words' in data:
                    predictions_list.append(data['top_words'])
                    
            except json.JSONDecodeError as e:
                print(f"Warning: Could not parse line {line_number}. Error: {str(e)}")
                print(f"Problematic line: {line[:100]}...")  # Print first 100 chars of the line
                continue
    
    return gaze_points_list, predictions_list

In [13]:
load_json_file("..\layout.txt")

{'keyboard': 'A (0.0000,0.2830)\nB (0.1628,0.2470)\nCD (0.2972,0.1510)\nEF (0.3800,0.0070)\nGH (0.3980,-0.1590)\nIJK (0.3464,-0.3170)\nL (0.2352,-0.4406)\nM (0.0804,-0.5082)\nNO (-0.0832,-0.5082)\nP (-0.2352,-0.4406)\nQR (-0.3464,-0.3170)\nS (-0.3980,-0.1590)\nTU (-0.3804,0.0066)\nVWX (-0.2972,0.1506)\nYZ (-0.1628,0.2486)',
 'center': {'x': -0.009399980306625366, 'y': -0.125},
 'inner_radius': 0.3230000138282776,
 'outer_radius': 0.5,
 'k': 1,
 'shape': 'circle',
 'left_bound': {'x': 0.0, 'y': 0.0},
 'right_bound': {'x': 0.0, 'y': 0.0},
 'top_bound': {'x': 0.0, 'y': 0.0},
 'bottom_bound': {'x': 0.0, 'y': 0.0}}

In [40]:
gaze_data, predictions = parse_gaze_and_predictions("..\eyeData\eyeTracking2024-10-09 12-11-54.txt")

In [41]:
print(predictions)

[['draft', 'craft', 'cure'], ['banana'], ['driving', 'diving', 'ring'], ['crisis', 'risks', 'disks'], ['eleven', 'even', 'ellen'], ['garage', 'grade', 'grace'], ['kinnock', 'knock', 'kind'], ['start', 'taut', 'saut'], ['relax', 'flaw', 'law'], ['work', 'wok', 'or'], ['stopped', 'stunned', 'topped'], ['window', 'widow', 'know'], ['moon', 'men', 'neo'], ['moment', 'menu', 'not'], ['shower', 'shove', 'hoover'], ['pursue', 'purse', 'true'], ['pledges', 'places', 'pledge'], ['plague', 'psyche', 'place'], ['candle', 'dance', 'angle'], ['shine', 'shoe', 'shin'], ['security', 'entity', 'equity'], ['medical', 'musical', 'music'], ['the', 'he'], ['leaf', 'leah', 'age'], ['crops', 'drops', 'clips'], ['tin', 'in', 'inn'], ['ton', 'on', 'no'], ['too', 'to'], ['the', 'tug', 'he'], ['ground', 'round', 'rooted'], ['the', 'tug', 'he'], ['night', 'fight', 'eight'], ['sky', 'i'], ['is', 'i'], ['bright', 'right', 'sight'], ['anshe', 'shed', 'aged'], ['need', 'and', 'anne'], ['clear', 'class', 'dear'], [],

In [37]:
base_url = "http://localhost:5000"  # Adjust port if needed
setup_endpoint = f"{base_url}/setup"
general_endpoint = f"{base_url}/general"
setup_data = load_json_file(Path("..\\tests\layout.txt"))

print("Making setup request...")
if not make_post_request(setup_endpoint, setup_data):
    print("Setup request failed. Exiting.")

predictions_with_language_model = []

for points in gaze_data:
    request_data = {
        "gaze_points": points
    }
    try:
        response = requests.post(general_endpoint, json=request_data)
        response.raise_for_status()  # Raise an exception for bad status codes
        predictions_with_language_model.append(response.json())  # Get the actual response data
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
        continue

Making setup request...
Success: POST to http://localhost:5000/setup
Response status code: 200
Response body: {
  "message": "Keyboard setup completed successfully"
}




In [38]:
predictions_with_language_model

[{'top_words': ['the', 'bug', 'beg']},
 {'top_words': ['beat', 'leaf', 'late']},
 {'top_words': ['learns', 'hearts', 'ghosts']},
 {'top_words': ['loops', 'lost', 'most']},
 {'top_words': ['sheet', 'shut', 'she']},
 {'top_words': ['deprived', 'prompted', 'chlorine']},
 {'top_words': ['huge', 'glue', 'glut']},
 {'top_words': ['ghost', 'joint', 'goins']},
 {'top_words': ['shitty', 'sixty', 'sight']},
 {'top_words': ['is', 'i']},
 {'top_words': ['benign', 'forgot', 'berlin']},
 {'top_words': ['wand', 'zane', 'med']},
 {'top_words': ['defeat', 'deity', 'delta']},
 {'top_words': ['a']},
 {'top_words': ['pools', 'polls', 'pals']},
 {'top_words': ['soybean', 'stream', 'strode']},
 {'top_words': ['fliers', 'killers', 'flops']},
 {'top_words': ['under', 'texts', 'modes']},
 {'top_words': ['the', 'he', 'fee']},
 {'top_words': ['bridge', 'hedge', 'ridge']},
 {'top_words': ['gentle', 'gene', 'home']},
 {'top_words': ['waves', 'wave', 'axes']},
 {'top_words': ['lagoon', 'hon', 'gop']},
 {'top_words'

In [39]:
for i in range(len(predictions_with_language_model)):
    print(predictions_with_language_model[i])
    print(predictions[i])

{'top_words': ['the', 'bug', 'beg']}
['the', 'eye', 'beef']
{'top_words': ['beat', 'leaf', 'late']}
['date', 'beat', 'deaf']
{'top_words': ['learns', 'hearts', 'ghosts']}
['hearts', 'learns', 'carrots']
{'top_words': ['loops', 'lost', 'most']}
['most', 'lost', 'loops']
{'top_words': ['sheet', 'shut', 'she']}
['shut', 'sheet', 'get']
{'top_words': ['deprived', 'prompted', 'chlorine']}
['prompted', 'deprived', 'chlorine']
{'top_words': ['huge', 'glue', 'glut']}
['huge', 'glue', 'glut']
{'top_words': ['ghost', 'joint', 'goins']}
['joint', 'ghost', 'input']
{'top_words': ['shitty', 'sixty', 'sight']}
['sight', 'shitty', 'sixty']
{'top_words': ['is', 'i']}
['is', 'i']
{'top_words': ['benign', 'forgot', 'berlin']}
['forgot', 'berlin', 'benign']
{'top_words': ['wand', 'zane', 'med']}
['wand', 'zane', 'wood']
{'top_words': ['defeat', 'deity', 'delta']}
['defeat', 'delta', 'delay']
{'top_words': ['a']}
['a']
{'top_words': ['pools', 'polls', 'pals']}
['polls', 'pools', 'malls']
{'top_words': ['s